# PCA with Scikit-Learn and Component Selection

Companion notebook for Sections 4.3 and 4.7 of the lecture notes.

Objectives:

- use `sklearn.decomposition.PCA`;
- inspect `components_`, `explained_variance_`, and `explained_variance_ratio_`;
- choose `n_components` using cumulative explained variance;
- compare PCA visualizations on Iris and Digits;
- treat the number of components as a modeling choice.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. PCA on Iris

Iris is small enough to inspect directly. Because its features have different units/scales, we standardize before PCA.


In [ ]:
from sklearn.datasets import load_iris, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target

iris_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
])
X_iris_pca = iris_pipe.fit_transform(X_iris)
pca_iris = iris_pipe.named_steps['pca']

print('PCA components shape:', pca_iris.components_.shape)
print('Explained variance ratio:', pca_iris.explained_variance_ratio_)
print('Cumulative explained variance:', pca_iris.explained_variance_ratio_.sum())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for class_id, class_name in enumerate(iris.target_names):
    mask = y_iris == class_id
    ax.scatter(X_iris_pca[mask, 0], X_iris_pca[mask, 1], label=class_name, alpha=0.8)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Iris projected onto two principal components')
ax.legend()
plt.show()


## 2. Interpreting loadings

Rows of `components_` are principal directions. Large absolute values indicate original features that contribute strongly to a component.


In [ ]:
loadings = pd.DataFrame(
    pca_iris.components_,
    columns=iris.feature_names,
    index=['PC1', 'PC2'],
)
loadings.T.sort_values('PC1', key=np.abs, ascending=False)


## 3. Choosing the number of components on Digits

Digits has 64 pixel features. A scree plot and cumulative explained variance show how many components retain a desired fraction of variance.


In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

X_digits_scaled = StandardScaler().fit_transform(X_digits)
pca_full = PCA().fit(X_digits_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(np.arange(1, len(explained) + 1), explained)
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Explained variance ratio')
axes[0].set_title('Scree plot')

axes[1].plot(np.arange(1, len(cumulative) + 1), cumulative, marker='o', markersize=3)
axes[1].axhline(0.90, color='red', linestyle='--', label='90%')
axes[1].axhline(0.95, color='green', linestyle='--', label='95%')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative explained variance')
axes[1].set_title('Cumulative explained variance')
axes[1].legend()
plt.tight_layout()
plt.show()

print('Components for 90%:', np.argmax(cumulative >= 0.90) + 1)
print('Components for 95%:', np.argmax(cumulative >= 0.95) + 1)


## 4. `n_components` as a variance threshold

If `n_components` is a float in `(0, 1)`, scikit-learn keeps as many components as needed to reach that explained-variance target.


In [ ]:
for threshold in [0.80, 0.90, 0.95, 0.99]:
    pca = PCA(n_components=threshold, svd_solver='full')
    X_reduced = pca.fit_transform(X_digits_scaled)
    print(f'threshold={threshold}: components={pca.n_components_:2d}, retained={pca.explained_variance_ratio_.sum():.3f}, shape={X_reduced.shape}')


## 5. Visualizing Digits with two components

A 2D PCA plot preserves the best linear variance structure. It is useful, but it should not be confused with nonlinear neighborhood-preserving embeddings such as t-SNE or UMAP.


In [ ]:
pca_2d = PCA(n_components=2)
X_digits_2d = pca_2d.fit_transform(X_digits_scaled)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_digits_2d[:, 0], X_digits_2d[:, 1], c=y_digits, cmap='tab10', s=12, alpha=0.8)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Digits projected with PCA')
plt.colorbar(scatter, ax=ax, label='digit')
plt.show()

print('2D retained variance:', pca_2d.explained_variance_ratio_.sum())


## 6. PCA as a modeling choice

For supervised learning, the number of components should often be selected by validation performance rather than explained variance alone.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for k in [2, 5, 10, 20, 30, 40, 0.90, 0.95, None]:
    steps = [('scaler', StandardScaler())]
    label = 'no PCA' if k is None else f'PCA {k}'
    if k is not None:
        steps.append(('pca', PCA(n_components=k, svd_solver='full')))
    steps.append(('clf', LogisticRegression(max_iter=2000)))
    pipe = Pipeline(steps)
    scores = cross_val_score(pipe, X_digits, y_digits, cv=cv, scoring='accuracy')
    rows.append({'setting': label, 'accuracy_mean': scores.mean(), 'accuracy_std': scores.std()})

pd.DataFrame(rows).sort_values('accuracy_mean', ascending=False)


## 7. Solvers and whitening

`randomized` SVD is useful when the dataset is large and only a small number of components is needed. `whiten=True` rescales component scores to unit variance; it can help some downstream models but also discards relative variance scale.


In [ ]:
for solver in ['full', 'randomized']:
    pca = PCA(n_components=20, svd_solver=solver, random_state=42)
    X_reduced = pca.fit_transform(X_digits_scaled)
    print(f'{solver:10s} retained variance: {pca.explained_variance_ratio_.sum():.4f}')

pca_plain = PCA(n_components=10).fit_transform(X_digits_scaled)
pca_white = PCA(n_components=10, whiten=True).fit_transform(X_digits_scaled)

print('\nVariance of first 5 plain component scores:', np.var(pca_plain[:, :5], axis=0, ddof=1))
print('Variance of first 5 whitened component scores:', np.var(pca_white[:, :5], axis=0, ddof=1))


## Takeaways

- `explained_variance_ratio_` supports scree plots and cumulative-variance thresholds.
- PCA components are linear combinations of original features.
- In predictive tasks, choose the number of components with validation when possible.
